# RAG问答实验: 从原理到实践的检索增强生成

> **学习目标**: 理解RAG架构原理，掌握向量数据库和Embedding，学会使用智析项目的 `rag_engine.py`
>
> **对应项目模块**: `src/rag_engine.py` (LLM应用层)
>
> **预计时间**: 3-4天
>
> **前置条件**: 
> - 已完成 Notebook 02 (文档解析)
> - `pip install langchain langchain-openai chromadb python-dotenv`
> - 配置好 `.env` 文件 (或安装Ollama)

---

## 背景知识: 什么是RAG？

### LLM的局限性
大语言模型（如GPT-4）虽然很强大，但有三个核心问题:
1. **知识截止**: 训练数据有截止日期，不知道最新信息
2. **幻觉**: 会编造看起来合理但实际错误的信息
3. **无法访问私有数据**: 不知道你的文档里写了什么

### RAG如何解决这些问题？
RAG (Retrieval-Augmented Generation) = **检索 + 生成**

```
用户提问 → 检索相关文档 → 将文档作为上下文 → 让LLM基于文档回答
```

**核心思想**: 不让LLM"凭空回答"，而是先找到相关信息，再让LLM基于这些信息生成答案。
这样就能:
- 回答最新信息 (只要文档是最新的)
- 减少幻觉 (有文档作为依据)
- 访问私有数据 (你自己的文档)

### RAG的5个核心步骤

| 步骤 | 名称 | 做什么 | 智析中的技术 |
|------|------|--------|-------------|
| 1 | Chunking | 将长文档切成小块 | `doc_parser.get_text_chunks()` |
| 2 | Embedding | 将文本块转换为向量 | OpenAI Embedding |
| 3 | Indexing | 将向量存入数据库 | ChromaDB |
| 4 | Retrieval | 用户提问时检索最相关的块 | 向量相似度搜索 |
| 5 | Generation | 将检索结果构建Prompt，LLM生成答案 | OpenAI/Ollama |

### 什么是Embedding（向量嵌入）？
Embedding是将文本转换为一个高维向量（如1536维的数字数组）。
- 语义相似的文本 → 向量距离近
- 语义不同的文本 → 向量距离远

例如:
- "人工智能"和"机器学习" → 向量很接近
- "人工智能"和"香蕉" → 向量很远

这就是为什么RAG能找到"语义相关"的文档，而不只是"关键词匹配"。

## Part 1: 理解Embedding — 向量化的魔力

> 注意: 下面的实验需要OpenAI API Key。如果你没有，可以跳过Part 1直接到Part 2。

In [ ]:
import os
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv()  # 从.env文件加载API Key

# 检查是否有API Key
has_api_key = bool(os.getenv('OPENAI_API_KEY'))
print(f"OpenAI API Key: {'已配置' if has_api_key else '未配置'}")

if has_api_key:
    # ========================================
    # 1.1 体验Embedding
    # ========================================
    from openai import OpenAI
    client = OpenAI()
    
    # 将文本转换为向量
    texts = [
        "人工智能正在改变世界",
        "机器学习是AI的核心技术",
        "今天天气真好，适合出去散步",
        "深度学习模型需要大量数据训练",
    ]
    
    embedding_model = os.getenv('EMBEDDING_MODEL', 'text-embedding-3-small')
    response = client.embeddings.create(input=texts, model=embedding_model)
    
    embeddings = [item.embedding for item in response.data]
    dim = len(embeddings[0])
    print(f"\nEmbedding维度: {dim}")  # 1536维
    print(f"第1个文本的向量前10个值: {embeddings[0][:10]}")
    
    # ========================================
    # 1.2 计算文本相似度
    # ========================================
    import numpy as np
    
    def cosine_similarity(a, b):
        """余弦相似度: 1=完全相同, 0=无关, -1=完全相反"""
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    
    print("\n=== 文本相似度矩阵 ===")
    print(f"{'':>30}", end="")
    for t in texts:
        print(f"{t[:12]:>15}", end="")
    print()
    
    for i, t1 in enumerate(texts):
        print(f"{t1[:12]:>30}", end="")
        for j, t2 in enumerate(texts):
            sim = cosine_similarity(embeddings[i], embeddings[j])
            print(f"{sim:>15.3f}", end="")
        print()
    
    print("\n观察: AI相关的文本之间相似度更高!")
else:
    print("\n跳过Embedding实验 (需要OpenAI API Key)")
    print("你可以:")
    print("  1. 在.env文件中配置OPENAI_API_KEY")
    print("  2. 或者安装Ollama使用本地模型")

## Part 2: ChromaDB — 向量数据库

### 什么是向量数据库？
传统数据库用SQL查询，向量数据库用"向量距离"查询。你给它一个向量，它能快速找到最相似的向量。

### 为什么用ChromaDB？
- **纯Python**: 不需要安装数据库服务器
- **嵌入式**: 直接在Python代码中运行
- **轻量**: 适合开发和小型项目
- **持久化**: 可以保存到磁盘，下次启动不用重建

In [ ]:
import chromadb

# ========================================
# 2.1 创建ChromaDB客户端和集合
# ========================================

# 使用内存模式 (不保存到磁盘，适合实验)
chroma_client = chromadb.Client()

# 创建一个集合 (collection) — 类似数据库中的"表"
collection = chroma_client.create_collection(
    name="test_docs",
    metadata={"description": "智析项目测试集合"}
)

print(f"集合已创建: {collection.name}")
print(f"当前文档数: {collection.count()}")

# ========================================
# 2.2 添加文档 (不需要Embedding模型也能测试!)
# ========================================
# ChromaDB有内置的Embedding函数 (默认用all-MiniLM-L6-v2)

# 模拟从PDF解析出的文档块
documents = [
    "智析是一个多模态文档智能分析平台，能解析PDF中的文字和图表。",
    "系统采用四层架构: 文档解析层、NLP分析层、知识构建层、RAG问答层。",
    "RAG代表检索增强生成，是当前最流行的LLM应用架构。",
    "ChromaDB是一个嵌入式向量数据库，纯Python实现，无需额外部署。",
    "LangChain是一个LLM应用开发框架，支持RAG、Agent等模式。",
    "PaddleOCR是百度开源的OCR工具，中文识别率非常高。",
    "知识图谱用NetworkX构建，能展示实体之间的关系。",
    "Streamlit可以在30行代码内创建一个Web应用。",
]

metadatas = [
    {"page": 1, "source": "项目介绍"},
    {"page": 1, "source": "项目介绍"},
    {"page": 2, "source": "RAG说明"},
    {"page": 2, "source": "技术说明"},
    {"page": 3, "source": "技术说明"},
    {"page": 1, "source": "技术说明"},
    {"page": 3, "source": "知识图谱"},
    {"page": 3, "source": "Web界面"},
]

ids = [f"doc_{i}" for i in range(len(documents))]

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
)

print(f"\n已添加 {collection.count()} 个文档块")
print(f"文档示例: {documents[0][:60]}...")

In [ ]:
# ========================================
# 2.3 语义搜索 — RAG的核心
# ========================================

# 提问: 找到最相关的文档块
queries = [
    "这个项目用了什么技术？",
    "什么是RAG？",
    "怎么识别图片中的文字？",
]

for query in queries:
    print(f"\n{'='*50}")
    print(f"查询: '{query}'")
    print(f"{'='*50}")
    
    results = collection.query(
        query_texts=[query],
        n_results=3,  # 返回最相关的3个文档
    )
    
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    )):
        print(f"\n  [{i+1}] 相关度距离: {dist:.4f} (越小越相关)")
        print(f"      来源: {meta['source']}, 页码: {meta['page']}")
        print(f"      内容: {doc}")

In [ ]:
# ========================================
# 2.4 带过滤条件的搜索
# ========================================

# 只搜索特定来源的文档
results = collection.query(
    query_texts=["项目架构是什么？"],
    n_results=5,
    where={"source": "项目介绍"},  # 只从"项目介绍"中搜索
)

print("=== 过滤搜索: 只搜索'项目介绍' ===")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"  [{i+1}] [{meta['source']}] {doc[:80]}")

# 查看所有文档
print(f"\n=== 集合中的所有文档 ({collection.count()}个) ===")
all_docs = collection.get()
for doc_id, doc, meta in zip(all_docs['ids'], all_docs['documents'], all_docs['metadatas']):
    print(f"  {doc_id}: [{meta['source']}] {doc[:60]}...")

## Part 3: 使用智析的 RAG 引擎

现在我们已经理解了底层原理，来使用项目中封装好的 `rag_engine.py`。

### `RAGEngine` 类的设计:
- **延迟初始化**: 模型在第一次使用时才加载，节省内存
- **双模式**: 支持OpenAI API和本地Ollama
- **完整链路**: `ingest_documents()` → `ask()` 两步完成

In [ ]:
# ========================================
# 3.1 完整RAG流程演示
# ========================================

from src.doc_parser import DocumentParser
from src.rag_engine import RAGEngine, RAGAnswer

# Step 1: 解析PDF并切块
print("[Step 1] 解析PDF...")
parser = DocumentParser(test_pdf_path, output_dir='../data/processed')
chunks = parser.get_text_chunks(chunk_size=500, chunk_overlap=50)
print(f"  切分为 {len(chunks)} 个块\n")

# Step 2: 初始化RAG引擎
print("[Step 2] 初始化RAG引擎...")

# 选择模式 (根据你的环境选择)
USE_OLLAMA = not has_api_key  # 没有API Key就用Ollama

if USE_OLLAMA:
    print("  使用 Ollama 本地模式")
    engine = RAGEngine(
        use_ollama=True,
        ollama_model="qwen2.5:7b",
    )
else:
    print("  使用 OpenAI API 模式")
    engine = RAGEngine(
        use_ollama=False,
        openai_model="gpt-4o-mini",  # 最便宜的模型
    )

# Step 3: 导入文档
print("\n[Step 3] 导入文档到向量数据库...")
engine.ingest_documents(chunks)

# Step 4: 提问!
print("\n[Step 4] 开始问答!")
questions = [
    "这个项目叫什么名字？它的主要功能是什么？",
    "系统采用了什么架构？有几层？",
    "RAG是什么？它的工作流程是怎样的？",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    answer = engine.ask(q, top_k=3)
    print(f"A: {answer.answer}")
    print(f"\n使用的模型: {answer.model_used}")
    print(f"参考来源数: {len(answer.sources)}")
    for src in answer.sources:
        print(f"  - 第{src['page']}页: {src['content'][:80]}...")

In [ ]:
# ========================================
# 3.2 纯检索模式 (不生成回答，只看检索结果)
# ========================================

print("=== 检索测试 (不消耗LLM tokens) ===\n")

search_queries = [
    "文本切块",
    "向量数据库",
    "知识图谱",
]

for query in search_queries:
    results = engine.search(query, top_k=2)
    print(f"查询: '{query}'")
    for i, r in enumerate(results):
        print(f"  [{i+1}] 第{r['page']}页: {r['content'][:100]}...")
    print()

## Part 4: 理解RAG Prompt工程

### 为什么Prompt很重要？
RAG的效果很大程度上取决于你如何构建Prompt。好的Prompt应该:
1. 明确告诉LLM"只基于文档回答"
2. 提供清晰的文档上下文
3. 允许LLM说"我不知道"

### 看看智析项目中的Prompt模板:

In [ ]:
# 查看RAGEngine使用的Prompt模板
from src.rag_engine import RAGEngine

engine = RAGEngine()

# 模拟一个问答场景
question = "ChromaDB有什么特点？"
context = """[文档片段 1]
ChromaDB是一个嵌入式向量数据库，纯Python实现，无需额外部署。
它支持持久化存储，可以将向量数据保存到磁盘。

[文档片段 2]
ChromaDB适合开发阶段和小型项目使用，
对于大规模生产环境，建议使用FAISS或Milvus。"""

# 查看生成的完整Prompt
prompt = engine._build_prompt(question, context)
print("=== RAG Prompt 模板 ===")
print(prompt)
print("\n" + "="*60)
print("观察: Prompt中包含了:")
print("  1. 系统指令 (只基于文档回答)")
print("  2. 检索到的文档内容")
print("  3. 用户问题")
print("  4. 明确的格式要求")

---
## 知识总结

### 你学到了什么:
1. **RAG原理**: 检索+生成，解决LLM的知识截止和幻觉问题
2. **Embedding**: 将文本转换为向量，实现语义搜索
3. **ChromaDB**: 嵌入式向量数据库，添加/查询/过滤
4. **RAGEngine**: 项目封装的RAG引擎，两步完成全流程
5. **Prompt工程**: RAG问答的Prompt模板设计

### RAG参数调优指南:
| 参数 | 作用 | 调优建议 |
|------|------|----------|
| `chunk_size` | 文本块大小 | 200-1000, 默认500 |
| `chunk_overlap` | 块间重叠 | 通常为chunk_size的10% |
| `top_k` | 检索数量 | 3-5, 太多会引入噪声 |
| `temperature` | 生成随机性 | 0.1(精确)到0.7(创造性) |

### 常见问题排查:
| 问题 | 可能原因 | 解决方案 |
|------|---------|---------|
| 回答不准确 | chunk_size太小 | 增大chunk_size |
| 回答"不知道" | 检索不到相关内容 | 增大top_k或调整切块策略 |
| 回答太慢 | 模型太大 | 换用gpt-4o-mini或量化模型 |
| API报错 | Key无效或余额不足 | 检查.env配置 |

### 下一步:
- 用**真实研报/论文**测试完整流程
- 运行 `04_nlp_analysis.ipynb` 学习NLP分析
- 运行 `05_knowledge_graph.ipynb` 构建知识图谱
- 运行 `streamlit run src/app.py` 体验Web界面